# 🛰️ NN_segmentation — Kaggle Launcher

Questo notebook è l'equivalente Kaggle del `Colab_Launcher.ipynb`.

---

## 📋 Setup iniziale (da fare UNA sola volta nel pannello Kaggle)

Prima di eseguire, collega i dataset dal pannello **"+ Add Data"** (barra laterale destra):

| Dataset Kaggle da allegare | Come trovarlo | Contiene |
|---|---|---|
| **OpenEarthMap** | Cerca: `global-land-cover-mapping-openearthmap` | Dataset raw già estratto ✅ |
| *(opzionale)* **OEM Indices** | Cerca: `oem-indices` (il tuo) | `oem_train_indices.json`, `oem_val_indices.json` |

> ✅ **Non devi caricare niente dal tuo PC!**  
> Il dataset OpenEarthMap è già disponibile pubblicamente su Kaggle.  
> Basta cercarlo e allegarlo come mostrato sopra.

---

## 📂 Struttura filesystem Kaggle
```
/kaggle/input/.../   ← read-only, allegato (rilevato in automatico)
    ├── images/train/*.tif    ← immagini RGB
    ├── images/val/*.tif
    ├── label/train/*.tif     ← mask (valori 0-8)
    ├── label/val/*.tif
    ├── train.txt             ← lista filename
    └── val.txt

/kaggle/input/oem-indices/          ← read-only, opzionale
    ├── oem_train_indices.json
    └── oem_val_indices.json

/kaggle/working/NN_segmentation/    ← read-write (output tab)
    ├── dataset/                    ← JSON indici attivi
    ├── best_model.pth
    ├── loss_curve.png
    └── output_samples/
```

---

## ♻️ Come riusare gli indici tra sessioni (risparmia ~15 min)
1. Dopo la prima run, vai su **"Output"** del notebook
2. Salva `oem_train_indices.json` e `oem_val_indices.json` come **nuovo Dataset Kaggle** chiamato `oem-indices`
3. Collegalo a questo notebook da **"+ Add Data"**  
4. Dalla 2ª volta la cella ④ dura <5 secondi

## ① Clone / Pull del codice da GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/robertopassante/NN_segmentation.git'
REPO_NAME = 'NN_segmentation'
REPO_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print(f'[INFO] Clonazione {REPO_NAME}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'[INFO] Repo già presente. Aggiornamento...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'[OK] Working directory: {os.getcwd()}')

## ② Installazione dipendenze e pesi pre-addestrati RSP Swin-T

In [ ]:
import os

!pip install -q -r requirements.txt
!pip install -q yacs safetensors rasterio

# Pesi RSP Swin-T (download solo se assenti)
WEIGHTS_PATH = '/kaggle/working/NN_segmentation/rsp-swin-t-ckpt.safetensors'
if not os.path.exists(WEIGHTS_PATH):
    print('[INFO] Download pesi RSP Swin-T...')
    !wget -q -O {WEIGHTS_PATH} \
        https://huggingface.co/BiliSakura/RSP-Swin-T/resolve/main/model.safetensors
    print('[OK] Pesi scaricati.')
else:
    print('[SKIP] Pesi RSP già presenti.')

## ③ Verifica dataset allegato

> Il dataset OpenEarthMap è già estratto in `/kaggle/input/`, non serve nessuna estrazione.
> Qui verifichiamo solo che sia stato allegato correttamente e troviamo l'indirizzo.

In [ ]:
import os
import glob

# ── Rilevamento automatico della cartella del dataset ────────────────────────
# Kaggle cambia spesso i path di montaggio (/kaggle/input/nome o /kaggle/input/datasets/autore/nome)
# Troviamo automaticamente il path cercando il file 'train.txt'
found_paths = glob.glob('/kaggle/input/**/train.txt', recursive=True)
if found_paths:
    INPUT_DIR = os.path.dirname(found_paths[0])
else:
    INPUT_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap' # Fallback

DATA_DIR   = '/kaggle/working/NN_segmentation/dataset'
os.makedirs(DATA_DIR, exist_ok=True)

# Verifica che il dataset sia allegato
if not os.path.exists(INPUT_DIR) or not os.path.exists(os.path.join(INPUT_DIR, 'train.txt')):
    print(f'❌ Dataset non trovato (path esplorato: {INPUT_DIR})')
    print(f'   → Vai su "+ Add Data" e cerca "global-land-cover-mapping-openearthmap"')
    raise FileNotFoundError(f'Dataset mancante: {INPUT_DIR}')

# Conta le immagini disponibili
for split in ['train', 'val']:
    img_dir = os.path.join(INPUT_DIR, 'images', split)
    lbl_dir = os.path.join(INPUT_DIR, 'label',  split)
    n_img = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_lbl = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'[{split}] Immagini: {n_img} | Mask: {n_lbl}')

print(f'\n✅ Dataset intercettato correttamente in: {INPUT_DIR}')

## ④ Preparazione Smart Subset (solo alla prima esecuzione)

> **Cosa fa:**  
> Legge ogni mask `.tif` con `rasterio`, trova la classe dominante (≥ 40%),  
> seleziona max 150 img/classe per train e 40/classe per val.  
> Salva i JSON in `/kaggle/working/` per poi esportarli come Dataset Kaggle.
>
> **Durata:** ~15-20 min la **prima** volta. Poi < 5 sec se `oem-indices` è allegato.

In [ ]:
import os, shutil

DATA_DIR      = '/kaggle/working/NN_segmentation/dataset'
WORK_DIR      = '/kaggle/working'
INDICES_INPUT = '/kaggle/input/oem-indices'   # dataset Kaggle opzionale

train_json = os.path.join(DATA_DIR, 'oem_train_indices.json')
val_json   = os.path.join(DATA_DIR, 'oem_val_indices.json')

# ── Prova a caricare da dataset Kaggle allegato ──────────────────────────────
if not (os.path.exists(train_json) and os.path.exists(val_json)):
    if os.path.exists(INDICES_INPUT):
        for fname in ['oem_train_indices.json', 'oem_val_indices.json']:
            src = os.path.join(INDICES_INPUT, fname)
            dst = os.path.join(DATA_DIR, fname)
            if os.path.exists(src):
                shutil.copy(src, dst)
                print(f'[OK]  {fname} caricato da "{INDICES_INPUT}"')

# ── Se ancora assenti: genera il subset ─────────────────────────────────────
if os.path.exists(train_json) and os.path.exists(val_json):
    print('\n✅ Indici già presenti — skip preparazione.')
else:
    print('🔍 Generazione smart subset (prima esecuzione, ~15-20 min)...')
    print('─' * 60)
    !python data/prepare_dataset_kaggle.py
    print('─' * 60)

    # Copia anche in /kaggle/working/ per renderli visibili nell'Output tab
    for fname in ['oem_train_indices.json', 'oem_val_indices.json']:
        src = os.path.join(DATA_DIR, fname)
        dst = os.path.join(WORK_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)

    print('\n✅ Subset pronto!')
    print('─' * 60)
    print('💡 Per riusarli nelle prossime sessioni (eviti 15 min di attesa):')
    print('   1. Tab "Output" → trovi oem_train_indices.json e oem_val_indices.json')
    print('   2. Crea un Dataset Kaggle privato chiamato "oem-indices" con quei file')
    print('   3. Allegalo a questo notebook da "+ Add Data"')
    print('─' * 60)

## ⑤ Training

In [ ]:
%cd /kaggle/working/NN_segmentation
!python main_kaggle.py

## ⑥ Esporta output nel tab "Output" di Kaggle

> I file vengono copiati in `/kaggle/working/` così appaiono nel tab **"Output"**  
> e possono essere scaricati o riusati in altri notebook.

In [ ]:
import os, shutil

REPO_DIR = '/kaggle/working/NN_segmentation'
OUT_DIR  = '/kaggle/working'

# Checkpoint e loss curve
for fname in ['best_model.pth', 'loss_curve.png']:
    src = os.path.join(REPO_DIR, fname)
    dst = os.path.join(OUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'[OK]  {fname} → {dst}')
    else:
        print(f'[SKIP] {fname} non trovato')

# Immagini di predizione per classe
pred_src = os.path.join(REPO_DIR, 'output_samples')
pred_dst = os.path.join(OUT_DIR,  'output_samples')
if os.path.exists(pred_src):
    shutil.copytree(pred_src, pred_dst, dirs_exist_ok=True)
    n = len(os.listdir(pred_dst))
    print(f'[OK]  output_samples/ ({n} immagini) → {pred_dst}')

print('\n✅ Output pronti nel tab "Output" di Kaggle.')